# PVELAD Faster R-CNN Training Pipeline

This notebook orchestrates the training using modular components:
- `voc_utils.py`: Logging and metrics
- `voc_aug.py`: Adaptive Augmentation
- `voc_dataset.py`: VOC Dataset loading

Configuration, Model Definition, and Training Engine are defined here for easy customization.

In [ ]:
import os
import torch
import numpy as np
import random
import math
import logging
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from collections import defaultdict

import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import DataLoader, WeightedRandomSampler
import torch.optim as optim

from rich.console import Console
from rich.panel import Panel
from rich.progress import Progress, SpinnerColumn, TextColumn, BarColumn, TaskProgressColumn, MofNCompleteColumn, TimeRemainingColumn

# Local Imports
from voc_utils import setup_rich_logging, MetricLogger, collate_fn
from voc_dataset import VOCSplitManager, VOCDataset
from voc_aug import AdaptiveAugmentation

In [ ]:
# Setup Logging
logger = setup_rich_logging()
console = Console()

## 1. Configuration

In [ ]:
@dataclass
class VOCConfig:
    """Central configuration for VOC dataset."""
    data_root: str = "trainval" # Default relative path for Kaggle
    img_size: int = 512
    mean: List[float] = field(default_factory=lambda: [0.485, 0.456, 0.406])
    std: List[float] = field(default_factory=lambda: [0.229, 0.224, 0.225])
    seed: int = 42
    num_workers: int = 2 # Reduced for stability
    models_dir: Path = Path("models")
    
    def __post_init__(self):
        self.data_root = Path(self.data_root).resolve()
        self.images_dir = self.data_root / "JPEGImages"
        self.annotations_dir = self.data_root / "Annotations"
        self.splits_dir = self.data_root / "ImageSets" / "Main"
        self.models_dir.mkdir(exist_ok=True, parents=True)

## 2. Model Definition

In [ ]:
def get_model(num_classes):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

## 3. Training Engine

In [ ]:
def train_one_epoch(model, optimizer, data_loader, device, epoch):
    model.train()
    metric_logger = MetricLogger(delimiter="  ")
    all_losses = []
    
    progress = Progress(
        SpinnerColumn(),
        TextColumn("[progress.description]{task.description}"),
        BarColumn(),
        TaskProgressColumn(),
        MofNCompleteColumn(),
        TimeRemainingColumn(),
        TextColumn("[bold blue]{task.fields[loss]}"),
    )
    task_id = progress.add_task(f"[cyan]Epoch {epoch} Training", total=len(data_loader), loss="Loss: 0.0000")
    
    with progress:
        for i, (images, targets) in enumerate(data_loader):
            images = list(image.to(device) for image in images)
            targets = [{k: v.to(device) for k, v in t.items() if k in ['boxes', 'labels']} for t in targets]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            
            if not math.isfinite(losses.item()):
                logger.error(f"Loss is {losses.item()}, skipping batch")
                continue

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()

            all_losses.append(losses.item())
            progress.update(task_id, advance=1, loss=f"Loss: {losses.item():.4f}")

    return np.mean(all_losses)

def evaluate(model, data_loader, device):
    model.train() # Keep in train mode for loss calculation
    losses = []
    progress = Progress(
        SpinnerColumn(), TextColumn("[progress.description]{task.description}"),
        BarColumn(), TaskProgressColumn(), MofNCompleteColumn()
    )
    task_id = progress.add_task("[green]Validating", total=len(data_loader))
    
    with progress:
        with torch.no_grad():
            for images, targets in data_loader:
                images = list(image.to(device) for image in images)
                targets = [{k: v.to(device) for k, v in t.items() if k in ['boxes', 'labels']} for t in targets]
                loss_dict = model(images, targets)
                losses.append(sum(l.item() for l in loss_dict.values()))
                progress.update(task_id, advance=1)
    return np.mean(losses)

## 4. Main Execution Pipeline

In [ ]:
# SETTINGS
EPOCHS = 15
BATCH_SIZE = 2 # Recommended for 3GB VRAM

console.print(Panel.fit("[bold magenta]PVELAD Kaggle Pipeline[/bold magenta]", border_style="magenta"))
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
console.print(f"[bold green]Using device:[/bold green] {device}")

# 1. Initialize Dataset
config = VOCConfig()
split_manager = VOCSplitManager(config, logger)
datasets = split_manager.create_splits()
train_dataset = datasets['train']
val_dataset = datasets['val']

# 2. Class Balancing (WeightedRandomSampler)
class_counts = train_dataset.class_distribution
total_samples = sum(class_counts.values())
weight_per_class = {k: total_samples/(v+1e-6) for k,v in class_counts.items()}

sample_weights = []
print("Calculating sample weights for balancing...")
for i in range(len(train_dataset)):
     # Direct access to dataset for speed
     _, target = train_dataset[i]
     labels = target['labels']
     if len(labels) > 0:
         # Assign weight based on the max weight class present in image (boosts minority)
         weight = max([weight_per_class.get(train_dataset.inv_class_map.get(int(l),''), 0) for l in labels])
     else:
         weight = 0
     sample_weights.append(weight)

num_classes = len(train_dataset.class_list) + 1
# Upsample to 1000 samples for quick testing or full size for real training
train_sampler = WeightedRandomSampler(sample_weights, len(train_dataset), replacement=True) 

# 3. Dataloaders
train_loader = DataLoader(train_dataset, BATCH_SIZE, sampler=train_sampler, collate_fn=collate_fn, num_workers=0)
val_loader = DataLoader(val_dataset, BATCH_SIZE, collate_fn=collate_fn, num_workers=0)

# 4. Model Setup
model = get_model(num_classes).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

# 5. Training Loop
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, optimizer, train_loader, device, epoch)
    val_loss = evaluate(model, val_loader, device)
    
    console.print(f"[bold]Epoch {epoch}[/bold] | Train: {train_loss:.4f} | Val: {val_loss:.4f}")
    
    # Save Best Model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), config.models_dir / "best_model.pth")
        console.print(f"[green]New best model saved! (Val Loss: {val_loss:.4f})[/green]")
        
    # Save Latest
    torch.save(model.state_dict(), config.models_dir / "latest.pth")